In [2]:
## Python v3.10.13

from __future__ import annotations
import numpy as np
import xlwings as xw
import pickle
import pandas as pd
import os
import sys
# sys.path.append('/home/wlh3/wagner')
from openai import AsyncOpenAI

from pydantic_ai import Agent
from openai import AsyncOpenAI
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from pydantic import BaseModel, Field, create_model
from typing import Dict, List, Any, Optional, Type, Literal
import asyncio
from enum import Enum
import json
import re
sys.path.append('/Users/williamharrigan/Desktop/test_wagner')
import creds

In [3]:
class AreAnntoationsEqual(BaseModel):
    are_equal: bool = Field(..., description="Are the two values synonymous?")
    # justification: str = Field(..., description="Justification of the propose value for are_equal. Justifications should be as concise as possible.")
    

## Setting the prompt and model for validation agent
validation_agent = Agent(
    # model="openai:o3-mini",
    model="openai:o4-mini",
    output_type=AreAnntoationsEqual,
    system_prompt = """You are an expert taxonomist. You are comparing the outcome of a manually extracted result versus an automatically extracted result. You need to compare the automatic results and determine whether the result is synonymous or equal the manual one; taking into consideration
    linguisitc and formatting nuances. If the measurements are correct but are seemigly in the wrong units, you can mark that as the results being the same = True. Your answer is whether the two results are similar True/False.""",
)

In [14]:
rows_to_validate = pd.read_csv('/Users/williamharrigan/Desktop/Github/ai_wagner_trait_data_extraction/extracted_test_3.csv')
rows_to_validate.head()

,family,genus,species,common_name,wagner_pg_number,description,infraspecific_epithet,stem_hair_type,phyllotaxy_type,breeding_type,...,seeds_perfruit,seed_length,seed_width,seed_diameter,pistillate_peduncle_length,pistillate_peduncle_width,staminate_pedicel_length,staminate_peduncle_length,staminate_peduncle_width,species_key
0,Apiaceae,Daucus,pusillus,American carrot,0,Dicots,NaN,['HISPID'],[],[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,apiaceae_daucus_pusillus
1,Apiaceae,Daucus,pusillus,American carrot,pg 203-204,Dicots,NaN,PUBERULENT,ALTERNATE,MONOECIOUS,...,"{'exmin': None, 'min': nan, 'max': nan, 'exmax...","{'exmin': nan, 'min': nan, 'max': nan, 'exmax'...","{'exmin': None, 'min': nan, 'max': nan, 'exmax...","{'exmin': None, 'min': nan, 'max': nan, 'exmax...","{'exmin': nan, 'min': nan, 'max': nan, 'exmax'...","{'exmin': nan, 'min': nan, 'max': nan, 'exmax'...","{'exmin': None, 'min': nan, 'max': nan, 'exmax...","{'exmin': nan, 'min': nan, 'max': nan, 'exmax'...","{'exmin': nan, 'min': nan, 'max': nan, 'exmax'...",apiaceae_daucus_pusillus
2,Apiaceae,Hydrocotyle,bowlesioides,Marsh pennywort,0,Dicots,NaN,['HIRSUTE'],[],[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,apiaceae_hydrocotyle_bowlesioides
3,Apiaceae,Hydrocotyle,bowlesioides,marsh pennywort,pg 205-206,Dicots,NaN,HIRSUTE,ALTERNATE,MONOECIOUS,...,"{'exmin': None, 'min': nan, 'max': nan, 'exmax...","{'exmin': nan, 'min': nan, 'max': nan, 'exmax'...","{'exmin': None, 'min': nan, 'max': nan, 'exmax...","{'exmin': None, 'min': nan, 'max': nan, 'exmax...","{'exmin': nan, 'min': nan, 'max': nan, 'exmax'...","{'exmin': nan, 'min': nan, 'max': nan, 'exmax'...","{'exmin': None, 'min': nan, 'max': nan, 'exmax...","{'exmin': nan, 'min': nan, 'max': nan, 'exmax'...","{'exmin': nan, 'min': nan, 'max': nan, 'exmax'...",apiaceae_hydrocotyle_bowlesioides
4,OLEACEAE,NESTEGIS,sandwicensis,Olive family,76,Dicots,NaN,[],['DECUSSATE'],[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,oleaceae_nestegis_sandwicensis


In [23]:
rows_to_validate

,family,genus,species,common_name,wagner_pg_number,description,infraspecific_epithet,stem_hair_type,phyllotaxy_type,breeding_type,...,seeds_perfruit,seed_length,seed_width,seed_diameter,pistillate_peduncle_length,pistillate_peduncle_width,staminate_pedicel_length,staminate_peduncle_length,staminate_peduncle_width,species_key
0,Apiaceae,Daucus,pusillus,American carrot,0,Dicots,NaN,['HISPID'],[],[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,apiaceae_daucus_pusillus
1,Apiaceae,Daucus,pusillus,American carrot,pg 203-204,Dicots,NaN,PUBERULENT,ALTERNATE,MONOECIOUS,...,"{'exmin': None, 'min': nan, 'max': nan, 'exmax...","{'exmin': nan, 'min': nan, 'max': nan, 'exmax'...","{'exmin': None, 'min': nan, 'max': nan, 'exmax...","{'exmin': None, 'min': nan, 'max': nan, 'exmax...","{'exmin': nan, 'min': nan, 'max': nan, 'exmax'...","{'exmin': nan, 'min': nan, 'max': nan, 'exmax'...","{'exmin': None, 'min': nan, 'max': nan, 'exmax...","{'exmin': nan, 'min': nan, 'max': nan, 'exmax'...","{'exmin': nan, 'min': nan, 'max': nan, 'exmax'...",apiaceae_daucus_pusillus
2,Apiaceae,Hydrocotyle,bowlesioides,Marsh pennywort,0,Dicots,NaN,['HIRSUTE'],[],[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,apiaceae_hydrocotyle_bowlesioides
3,Apiaceae,Hydrocotyle,bowlesioides,marsh pennywort,pg 205-206,Dicots,NaN,HIRSUTE,ALTERNATE,MONOECIOUS,...,"{'exmin': None, 'min': nan, 'max': nan, 'exmax...","{'exmin': nan, 'min': nan, 'max': nan, 'exmax'...","{'exmin': None, 'min': nan, 'max': nan, 'exmax...","{'exmin': None, 'min': nan, 'max': nan, 'exmax...","{'exmin': nan, 'min': nan, 'max': nan, 'exmax'...","{'exmin': nan, 'min': nan, 'max': nan, 'exmax'...","{'exmin': None, 'min': nan, 'max': nan, 'exmax...","{'exmin': nan, 'min': nan, 'max': nan, 'exmax'...","{'exmin': nan, 'min': nan, 'max': nan, 'exmax'...",apiaceae_hydrocotyle_bowlesioides
4,OLEACEAE,NESTEGIS,sandwicensis,Olive family,76,Dicots,NaN,[],['DECUSSATE'],[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,oleaceae_nestegis_sandwicensis
5,Oleaceae,Nestegis,sandwicensis,olopua,pg 990NA2,Dicots,NaN,GLABROUS,DECUSSATE,MONOECIOUS or DIOECIOUS,...,"{'exmin': None, 'min': nan, 'max': nan, 'exmax...","{'exmin': nan, 'min': nan, 'max': nan, 'exmax'...","{'exmin': None, 'min': nan, 'max': nan, 'exmax...","{'exmin': None, 'min': nan, 'max': nan, 'exmax...","{'exmin': nan, 'min': nan, 'max': nan, 'exmax'...","{'exmin': nan, 'min': nan, 'max': nan, 'exmax'...","{'exmin': None, 'min': nan, 'max': nan, 'exmax...","{'exmin': nan, 'min': nan, 'max': nan, 'exmax'...","{'exmin': nan, 'min': nan, 'max': nan, 'exmax'...",oleaceae_nestegis_sandwicensis
6,Onagraceae,Ludwigia,palustris,Marsh purslane,77,Dicots,pacifica,['GLABROUS'],['OPPOSITE'],[],...,NaN,"{'exmin': None, 'min': 0.6, 'max': 0.9, 'exmax...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,onagraceae_ludwigia_palustris
7,Onagraceae,Ludwigia,palustris,marsh purslane,pg 997NA,Dicots,NaN,GLABROUS,OPPOSITE,MONOECIOUS,...,"{'exmin': None, 'min': nan, 'max': nan, 'exmax...","{'exmin': nan, 'min': 0.06, 'max': 0.09, 'exma...","{'exmin': None, 'min': nan, 'max': nan, 'exmax...","{'exmin': None, 'min': nan, 'max': nan, 'exmax...","{'exmin': nan, 'min': nan, 'max': nan, 'exmax'...","{'exmin': nan, 'min': nan, 'max': nan, 'exmax'...","{'exmin': None, 'min': nan, 'max': nan, 'exmax...","{'exmin': nan, 'min': nan, 'max': nan, 'exmax'...","{'exmin': nan, 'min': nan, 'max': nan, 'exmax'...",onagraceae_ludwigia_palustris
8,Apiaceae,Spermolepis,hawaiiensis,NaN,0,Dicots,NaN,['GLABROUS'],[],[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,apiaceae_spermolepis_hawaiiensis
9,Apiaceae,Spermolepis,hawaiiensis,no common name,pg 210-212,Dicots,NaN,GLABROUS,ALTERNATE,MONOECIOUS,...,"{'exmin': None, 'min': nan, 'max': nan, 'exmax...","{'exmin': nan, 'min': nan, 'max': nan, 'exmax'...","{'exmin': None, 'min': nan, 'max': nan, 'exmax...","{'exmin': None, 'min': nan, 'max': nan, 'exmax...","{'exmin': nan, 'min': nan, 'max': nan, 'exmax'...","{'exmin': nan, 'min': nan, 'max': nan, 'exmax'...","{'exmin': None, 'min': nan, 'max': nan, 'ex

In [19]:
import math

def _is_empty(val):
    if val is None:
        return True
    if isinstance(val, float) and math.isnan(val):
        return True
    if isinstance(val, str) and val.strip().lower() in ("nan", "none", ""):
        return True
    return False

async def _compare_column(col, auto_val, manual_val):
    """Return (col, code): 0=incorrect, 1=correct, 2=needs manual review."""
    auto_empty = _is_empty(auto_val)
    manual_empty = _is_empty(manual_val)

    if auto_empty and manual_empty:
        return col, 1
    if auto_empty != manual_empty:
        return col, 2

    result = await validation_agent.run(f"Manual: {manual_val} Automatic: {auto_val}")
    return col, (1 if result.output.are_equal else 0)

async def build_validation_df(df, skip_cols=None):
    """
    Compare auto vs manual rows and return a 3-row-per-species_key DataFrame.

    Input df: 2 rows per species_key — row 1 = auto extracted, row 2 = manual extracted.

    Output row order per species_key:
      1. manual extracted   (row_type = "Manual")
      2. auto extracted     (row_type = "Automatic")
      3. coded comparison   (row_type = "IsCorrect"; values: 0=incorrect, 1=correct, 2=needs review)

    Args:
        df:         DataFrame with 2 rows per species_key
        skip_cols:  Columns excluded from agent comparison (copied as-is to coded row)
    """
    if skip_cols is None:
        skip_cols = {"species_key"}

    compare_cols = [c for c in df.columns if c not in skip_cols]
    result_rows = []

    for species_key, group in df.groupby("species_key", sort=False):
        if len(group) != 2:
            print(f"Warning: {species_key} has {len(group)} rows, expected 2 — skipping.")
            continue

        auto_row   = group.iloc[0]
        manual_row = group.iloc[1]

        col_results = await asyncio.gather(
            *[_compare_column(col, auto_row[col], manual_row[col]) for col in compare_cols]
        )

        coded_row = {col: None for col in df.columns}
        for col in skip_cols:
            if col in df.columns:
                coded_row[col] = auto_row[col]
        for col, code in col_results:
            coded_row[col] = code

        result_rows.extend([
            {"row_type": "Manual",     **manual_row.to_dict()},
            {"row_type": "Automatic",  **auto_row.to_dict()},
            {"row_type": "IsCorrect",  **coded_row},
        ])

    out_cols = ["row_type"] + df.columns.tolist()
    return pd.DataFrame(result_rows, columns=out_cols)

In [20]:
# Run comparison
validation_df = await build_validation_df(rows_to_validate)
validation_df.head()

,row_type,family,genus,species,common_name,wagner_pg_number,description,infraspecific_epithet,stem_hair_type,phyllotaxy_type,...,seeds_perfruit,seed_length,seed_width,seed_diameter,pistillate_peduncle_length,pistillate_peduncle_width,staminate_pedicel_length,staminate_peduncle_length,staminate_peduncle_width,species_key
0,Manual,Apiaceae,Daucus,pusillus,American carrot,pg 203-204,Dicots,NaN,PUBERULENT,ALTERNATE,...,"{'exmin': None, 'min': nan, 'max': nan, 'exmax...","{'exmin': nan, 'min': nan, 'max': nan, 'exmax'...","{'exmin': None, 'min': nan, 'max': nan, 'exmax...","{'exmin': None, 'min': nan, 'max': nan, 'exmax...","{'exmin': nan, 'min': nan, 'max': nan, 'exmax'...","{'exmin': nan, 'min': nan, 'max': nan, 'exmax'...","{'exmin': None, 'min': nan, 'max': nan, 'exmax...","{'exmin': nan, 'min': nan, 'max': nan, 'exmax'...","{'exmin': nan, 'min': nan, 'max': nan, 'exmax'...",apiaceae_daucus_pusillus
1,Automatic,Apiaceae,Daucus,pusillus,American carrot,0,Dicots,NaN,['HISPID'],[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,apiaceae_daucus_pusillus
2,IsCorrect,1,1,1,1,0,1,1,0,0,...,2,2,2,2,2,2,2,2,2,apiaceae_daucus_pusillus
3,Manual,Apiaceae,Hydrocotyle,bowlesioides,marsh pennywort,pg 205-206,Dicots,NaN,HIRSUTE,ALTERNATE,...,"{'exmin': None, 'min': nan, 'max': nan, 'exmax...","{'exmin': nan, 'min': nan, 'max': nan, 'exmax'...","{'exmin': None, 'min': nan, 'max': nan, 'exmax...","{'exmin': None, 'min': nan, 'max': nan, 'exmax...","{'exmin': nan, 'min': nan, 'max': nan, 'exmax'...","{'exmin': nan, 'min': nan, 'max': nan, 'exmax'...","{'exmin': None, 'min': nan, 'max': nan, 'exmax...","{'exmin': nan, 'min': nan, 'max': nan, 'exmax'...","{'exmin': nan, 'min': nan, 'max': nan, 'exmax'...",apiaceae_hydrocotyle_bowlesioides
4,Automatic,Apiaceae,Hydrocotyle,bowlesioides,Marsh pennywort,0,Dicots,NaN,['HIRSUTE'],[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,apiaceae_hydrocotyle_bowlesioides


In [21]:
def export_validation_to_excel(validation_df, filepath, sheet_name="Validation"):
    """
    Write validation_df to Excel and color all 3 rows of each column triplet
    by the coded row value.

    Colors:
      Green  (code=1) — correct
      Red    (code=0) — incorrect
      Yellow (code=2) — needs manual review

    Args:
        validation_df:  Output of build_validation_df (3 rows per species_key)
        filepath:       Path to save the .xlsx file
        sheet_name:     Sheet name (default "Validation")
    """
    COLOR_MAP = {
        0: (255, 102, 102),
        1: (102, 204, 102),
        2: (255, 255, 102),
    }

    skip_cols = {
        "row_type", "family", "genus", "species", "common_name", "description",
        "infraspecific_epithet", "species_key"
    }

    col_indices = {col: i + 1 for i, col in enumerate(validation_df.columns)}

    app = xw.App(visible=False)
    try:
        wb = app.books.add()
        ws = wb.sheets[0]
        ws.name = sheet_name

        ws.range("A1").value = [validation_df.columns.tolist()] + validation_df.values.tolist()

        n_rows = len(validation_df)
        assert n_rows % 3 == 0, "validation_df must have 3 rows per species_key"

        for triplet_start in range(0, n_rows, 3):
            coded_row = validation_df.iloc[triplet_start + 2]
            xl_row1 = triplet_start + 2
            xl_row3 = triplet_start + 4

            for col in validation_df.columns:
                if col in skip_cols:
                    continue
                try:
                    code = int(coded_row[col])
                except (TypeError, ValueError):
                    continue
                if code not in COLOR_MAP:
                    continue

                ws.range(
                    (xl_row1, col_indices[col]),
                    (xl_row3, col_indices[col])
                ).color = COLOR_MAP[code]

        wb.save(filepath)
        wb.close()
        print(f"Saved: {filepath}")
    finally:
        app.quit()

In [22]:
export_validation_to_excel(validation_df, 'validation_output.xlsx')

Saved: validation_output.xlsx
